In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
#Checking the dataset file paths
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    print(dirname)


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

#Set dataset paths
DATA_ROOT = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"

train_path = os.path.join(DATA_ROOT, "train")
val_path = os.path.join(DATA_ROOT, "val")
test_path = os.path.join(DATA_ROOT, "test")


#Verify folders
print("Training folders:", os.listdir(train_path))
print("Validation folders:", os.listdir(val_path))
print("Test folders:", os.listdir(test_path))


In [ ]:
#Count images in each folder
for split_name, split_path in {
    "Training": train_path,
    "Validation": val_path,
    "Test": test_path
}.items():
    print(f"\n{split_name} Set:")
    for class_name in os.listdir(split_path):
        class_path = os.path.join(split_path, class_name)
        print(f"{class_name}: {len(os.listdir(class_path))} images")


In [ ]:
#Load image datasets from the train, validation, and test directories.
#The folder names become the class labels.
#Since this is a binary classification problem, label_mode="binary" is used.

BATCH_SIZE = 32
IMG_SIZE = (224, 224)

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)


In [ ]:
#Import evaluation metrics used to assess model performance beyond accuracy.
#These metrics are especially important for pneumonia detection because
#false negatives represent pneumonia cases incorrectly predicted as normal.

from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
#Define a data augmentation pipeline.
#These random image transformations are applied during training to make the
#model more robust and reduce overfitting. The goal is to help the CNN learn general image patterns instead of memorizing the exact training images.

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="augmentation")
#Define a helper function to visualize training progress.
#This plots training vs. validation loss and accuracy across epochs.
#The plots help show whether the model is learning, improving, or overfitting.

def plot_history(history, title="Training"):
    plt.figure()
    plt.plot(history.history["loss"], label="train_loss")
    plt.plot(history.history["val_loss"], label="val_loss")
    plt.title(title + " - Loss")
    plt.legend()
    plt.show()

    plt.figure()
    plt.plot(history.history.get("accuracy", []), label="train_acc")
    plt.plot(history.history.get("val_accuracy", []), label="val_acc")
    plt.title(title + " - Accuracy")
    plt.legend()
    plt.show()


In [ ]:
def evaluate_model(model, test_ds, threshold=0.5):
    y_true = []
    y_prob = []
    for x, y in test_ds:
        probs = model.predict(x, verbose=0).ravel()
        y_prob.extend(probs)
        y_true.extend(y.numpy().ravel())
    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    print("Confusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(
        y_true,
        y_pred,
        target_names=["NORMAL", "PNEUMONIA"]
    ))
    auc = roc_auc_score(y_true, y_prob)
    print("ROC AUC:", auc)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
    plt.plot([0, 1], [0, 1], "--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.show()
    return {
        "confusion_matrix": cm,
        "auc": auc
    }


Model 1: CNN Built from Scratch¶


In [ ]:
from tensorflow.keras import layers, models

#Normalize pixel values from [0,255] to [0,1]
normalization = layers.Rescaling(1./255)

scratch_model = models.Sequential([
    #Input layer specifying image shape
    layers.Input(shape=IMG_SIZE + (3,)),

    #Apply data augmentation only during training
    data_augmentation,

    #Normalize images
    normalization,

    #Convolutional block 1
    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),  #Learn low-level features (edges, textures)
    layers.MaxPooling2D(),                                         #Reduce spatial dimensions

    #Convolutional block 2
    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),  #Learn more complex patterns
    layers.MaxPooling2D(),

    #Convolutional block 3
    layers.Conv2D(128, (3, 3), padding="same", activation="relu"), #Learn higher-level features
    layers.MaxPooling2D(),

    #Convolutional block 4
    layers.Conv2D(256, (3, 3), padding="same", activation="relu"), #Capture abstract spatial features
    layers.GlobalAveragePooling2D(),                               #Reduce parameters vs Flatten
    #Fully connected layers
    layers.Dropout(0.3),                                           #Reduce overfitting
    layers.Dense(128, activation="relu"),                          #Learn decision-level representations
    layers.Dropout(0.3),

    # Output layer for binary classification
    layers.Dense(1, activation="sigmoid")                          #Probability of pneumonia
], name="Scratch_CNN")
scratch_model.summary()


In [ ]:
scratch_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),        #Overall separability
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")   #VERY important for medical imaging (false negatives)
    ]
)


Train the Baseline CNN
This section trains the CNN built from scratch using the training dataset and evaluates it against the validation dataset after each epoch. Although the model is allowed to train for up to 50 epochs, several callbacks are used to make training more efficient and reduce overfitting.

Early stopping stops training if validation performance stops improving, while model checkpointing saves the best version of the model during training. The learning rate scheduler reduces the learning rate when progress slows, allowing the model to continue improving with smaller weight updates. TensorBoard logging is also included so training progress can be reviewed visually if needed.



In [ ]:
#Train the from-scratch CNN.
#This is the step where the model learns from the training images.
#The validation set is used after each epoch to monitor generalization performance.
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        patience=5,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        patience=2,
        factor=0.3,
        min_lr=1e-6
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "scratch_best.keras",
        save_best_only=True
    ),

    tf.keras.callbacks.TensorBoard(
        log_dir="logs/scratch"
    )
]

history_scratch = scratch_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=1,
    callbacks=callbacks
)

In [ ]:
#Plot the training and validation loss/accuracy curves for the scratch CNN.
#This helps visualize how the model learned over time and whether it may have overfit.

plot_history(history_scratch, title="Scratch CNN")


In [ ]:
#Evaluate the trained scratch CNN on the test dataset.
#The results include the confusion matrix, classification report, ROC AUC, and ROC curve

scratch_results = evaluate_model(scratch_model, test_ds)


Model 2: Transfer Learning with MobileNetV2¶


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

# ===========================================
# Image size
# ===========================================
IMG_SIZE = (224, 224)

# ===========================================
# Data Augmentation
# ===========================================
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# ===========================================
# Load MobileNetV2
# ===========================================
try:
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights="imagenet"
    )
    print("✅ Loaded pretrained ImageNet weights.")

except Exception as e:
    print("⚠ Could not download ImageNet weights.")
    print(e)
    print("Loading MobileNetV2 without pretrained weights...")

    base_model = tf.keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights=None
    )

# ===========================================
# Freeze Base Model
# ===========================================
base_model.trainable = False

# ===========================================
# Build Transfer Learning Model
# ===========================================
inputs = tf.keras.Input(shape=IMG_SIZE + (3,))

x = data_augmentation(inputs)

x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    1,
    activation="sigmoid"
)(x)

transfer_model = tf.keras.Model(
    inputs,
    outputs,
    name="MobileNetV2_Transfer_CNN"
)

# ===========================================
# Compile Model
# ===========================================
transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# ===========================================
# Model Summary
# ===========================================
transfer_model.summary()

In [ ]:
transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)
callbacks_transfer = [
    #Stop training if validation loss does not improve for 5 epochs.
    #Restore the weights from the best validation epoch.
    tf.keras.callbacks.EarlyStopping(
        patience=5,
        restore_best_weights=True
    ),

    #Reduce the learning rate when validation loss stops improving.
    #This allows the model to continue learning with smaller updates.
    tf.keras.callbacks.ReduceLROnPlateau(
        patience=2,
        factor=0.3,
        min_lr=1e-6
    ),

    #Save the best frozen-transfer model during training.
    tf.keras.callbacks.ModelCheckpoint(
        "transfer_stage1_best.keras",
        save_best_only=True
    ),

    #Save training logs for TensorBoard visualization.
    tf.keras.callbacks.TensorBoard(
        log_dir="logs/transfer_stage1"
    )
]


In [ ]:
#Train the transfer learning model.
#Since the MobileNetV2 backbone is frozen, training focuses on the new classifier head.
history_transfer_stage1 = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=1,
    callbacks=callbacks_transfer
)


In [ ]:
#Plot the training and validation loss/accuracy curves for the frozen transfer learning model.
#This helps compare its training behavior against the CNN built from scratch.
plot_history(history_transfer_stage1, title="Frozen MobileNetV2")


In [ ]:
#Evaluate the frozen MobileNetV2 transfer learning model on the test dataset.
#These results will be compared against the scratch CNN baseline.
transfer_stage1_results = evaluate_model(transfer_model, test_ds)


Model 3: Fine-Tune the Transfer Learning Model¶


In [ ]:
#Fine-tune the transfer learning model.
#The model has already trained a new classifier head while MobileNetV2 was frozen.
#Now, some of the later MobileNetV2 layers will be unfrozen so they can adapt more specifically to the chest X-ray pneumonia classification task.

#Unfreeze the MobileNetV2 base model.
base_model.trainable = True

#Keep the earlier layers frozen and fine-tune only the later layers.
#Earlier layers usually learn general features like edges and textures.
#Later layers learn more task-specific patterns, so they are better suited for fine-tuning.
fine_tune_at = int(len(base_model.layers) * 0.7)

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

#Recompile the model with a much smaller learning rate.
#A small learning rate is important during fine-tuning because the pretrained weights should be updated carefully rather than changed too aggressively.
transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)
#Define callbacks for fine-tuning.
#These help prevent overfitting, reduce the learning rate when progress slows, save the best fine-tuned model, and record logs for TensorBoard.
callbacks_ft = [
    tf.keras.callbacks.EarlyStopping(
        patience=5,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        patience=2,
        factor=0.3,
        min_lr=1e-7
    ),

    tf.keras.callbacks.ModelCheckpoint(
        "transfer_finetuned_best.keras",
        save_best_only=True
    ),

    tf.keras.callbacks.TensorBoard(
        log_dir="logs/transfer_finetune"
    )
]


In [ ]:
#Train the fine-tuned transfer learning model.
#During this stage, the classifier head and the later MobileNetV2 layers can learn.
history_transfer_ft = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=1,
    callbacks=callbacks_ft
)


In [ ]:
#Plot the training and validation loss/accuracy curves for the fine-tuned transfer learning model.
#This helps show whether unfreezing later MobileNetV2 layers improved training behavior.
plot_history(history_transfer_ft, title="Fine-Tuned MobileNetV2")


In [ ]:
#Evaluate the fine-tuned MobileNetV2 model on the test dataset.
#These results will be compared against both the scratch CNN and frozen transfer learning model.
transfer_ft_results = evaluate_model(transfer_model, test_ds)


In [ ]:
#Create a final model comparison table using the test-set results
comparison_df = pd.DataFrame({
    "Model": [
        "Scratch CNN",
        "Frozen MobileNetV2",
        "Fine-Tuned MobileNetV2"
    ],
    "Accuracy": [0.80, 0.75, 0.87],
    "ROC AUC": [0.881, 0.938, 0.971],
    "Pneumonia Recall": [0.75, 0.99, 0.98],
    "Normal Recall": [0.88, 0.34, 0.69],
    "False Negatives": [97, 3, 6],
    "False Positives": [27, 154, 73]
})

comparison_df


In [ ]:
#Plot accuracy, AUC, pneumonia recall, and normal recall for each model
metrics_to_plot = ["Accuracy", "ROC AUC", "Pneumonia Recall", "Normal Recall"]

ax = comparison_df.set_index("Model")[metrics_to_plot].plot(
    kind="bar",
    figsize=(12, 6),
    rot=0
)

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.legend(title="Metric", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(axis="y", alpha=0.3)
plt.show()


In [ ]:
#Plot false negatives and false positives for each model
error_metrics = ["False Negatives", "False Positives"]

ax = comparison_df.set_index("Model")[error_metrics].plot(
    kind="bar",
    figsize=(10, 6),
    rot=0
)

plt.title("Model Error Comparison")
plt.ylabel("Number of Images")
plt.legend(title="Error Type")
plt.grid(axis="y", alpha=0.3)
plt.show()
